# 5. Line buffers: read each pixel once

The naive kernel's sin was re-reading. A **line buffer** fixes it: keep the previous two
rows of the image in on-chip BRAM, so as each new pixel streams in it instantly completes
a vertical column of the 3x3 window - every pixel is read from DRAM **exactly once**.
With the window held in registers (all nine taps available the same cycle, nine DSPs),
the compute loop now hits **II=1**: one output pixel per clock. This is `conv3x3_fast.cpp`.

In *compute* terms this is the fast path. But watch what becomes the bottleneck next.

> **Board only.** The rungs below need the PYNQ board (`pynq` + the bitstream). On a
> laptop `available_backends()` omits them and the cell prints a skip message - that is
> expected; run this one on the board.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from fpga_conv import KERNELS, get_kernel, conv_reference, to_grayscale_u8, available_backends
from fpga_conv.bench import (
    Scoreboard, run_rung, time_backend, sample_gray, make_sample_image, roofline,
)

# Path to the combined bitstream (carries all four accelerators). Its .hwh must sit
# beside it. Relative to this notebook on the dev box; on the board point it at wherever
# you copied the build, e.g. '/home/xilinx/workshop/conv3x3.bit'.
BIT_PATH = os.path.abspath('../../rtl/build/conv3x3.bit')
HW_KW = {'bitfile': BIT_PATH}          # passed to the hardware rungs only

# The scoreboard persists across the whole series (notebooks/scoreboard.json), so the
# roofline grows as you climb. The hardware rungs are skipped off the board.
sb = Scoreboard()
print('backends available here:', available_backends())

In [ ]:
if 'hls_opt' in available_backends():
    img = sample_gray(256)
    out, t = run_rung('hls_opt', img, 'edges', repeats=10, scoreboard=sb, **HW_KW)
else:
    print('hls_opt backend not available here (laptop) - run this cell on the board.')

Better - and on the roofline this dot moves **right**: each pixel is now read once and
written once (2 bytes/pixel instead of 10), so the arithmetic intensity went up. The
compute loop is II=1.

So why isn't it ten times faster? Because the win exposed the *next* wall. The
accelerator's memory port is byte-wide, so even reading each pixel just once, it can only
pull in one byte per clock - and it still has to write a byte out. The datapath that can
finish a pixel every clock is now waiting on a memory interface that delivers data a byte
at a time. **We are memory-bandwidth bound** (~10 Mpix/s on this board). The dot sits
just under the memory-bandwidth ceiling.

In [ ]:
sb.roofline(); plt.show()

**The fix:** stop trickling bytes. Read and write in *bursts*, and overlap reading,
computing, and writing so memory traffic hides behind compute. -> `06_streaming.ipynb`.